In [15]:
from collections import defaultdict, deque
import copy


class SudokuCSP:
    def __init__(self, board):
        self.board = board
        self.variables = [(r, c) for r in range(9) for c in range(9)]

        # domains
        self.domains = {}
        for r in range(9):
            for c in range(9):
                if board[r][c] == 0:
                    self.domains[(r, c)] = set(range(1, 10))
                else:
                    self.domains[(r, c)] = {board[r][c]}

        self.neighbors = self.build_neighbors()

        self.backtrack_calls = 0
        self.backtrack_failures = 0

    # BUILD NEIGHBORS (row, col, box)
    def build_neighbors(self):
        neighbors = defaultdict(set)

        for r in range(9):
            for c in range(9):

                # row + column
                for k in range(9):
                    if k != c:
                        neighbors[(r, c)].add((r, k))
                    if k != r:
                        neighbors[(r, c)].add((k, c))

                # 3x3 box
                br, bc = 3 * (r // 3), 3 * (c // 3)
                for i in range(br, br + 3):
                    for j in range(bc, bc + 3):
                        if (i, j) != (r, c):
                            neighbors[(r, c)].add((i, j))

        return neighbors

    # AC-3 ALGORITHM
    def ac3(self):
        queue = deque()

        for xi in self.variables:
            for xj in self.neighbors[xi]:
                queue.append((xi, xj))

        while queue:
            xi, xj = queue.popleft()

            if self.revise(xi, xj):
                if len(self.domains[xi]) == 0:
                    return False

                for xk in self.neighbors[xi]:
                    if xk != xj:
                        queue.append((xk, xi))

        return True

    def revise(self, xi, xj):
        removed = False
        to_remove = set()

        for x in self.domains[xi]:
            if not any(x != y for y in self.domains[xj]):
                to_remove.add(x)

        if to_remove:
            self.domains[xi] -= to_remove
            removed = True

        return removed

    # MRV HEURISTIC
    def select_unassigned(self, assignment):
        unassigned = [v for v in self.variables if v not in assignment]
        return min(unassigned, key=lambda v: len(self.domains[v]))

    # CONSISTENCY CHECK
    def is_consistent(self, var, value, assignment):
        for n in self.neighbors[var]:
            if n in assignment and assignment[n] == value:
                return False
        return True

    # BACKTRACKING SEARCH
    def backtrack(self, assignment):
        self.backtrack_calls += 1

        if len(assignment) == 81:
            return assignment

        var = self.select_unassigned(assignment)

        for value in sorted(self.domains[var]):

            if self.is_consistent(var, value, assignment):

                old_domains = copy.deepcopy(self.domains)

                assignment[var] = value
                self.domains[var] = {value}

                if self.ac3():
                    result = self.backtrack(assignment)
                    if result:
                        return result

                # rollback
                self.domains = old_domains
                assignment.pop(var, None)

        self.backtrack_failures += 1
        return None

    # SOLVE
    def solve(self):
        self.ac3()
        return self.backtrack({})


# LOAD BOARD
def load_board(filename):
    board = []
    with open(filename, "r") as f:
        for line in f:
            board.append([int(x) for x in line.strip()])
    return board


# PRINT BOARD (3x3 GRID FORMAT)
def print_board(board):
    print("+-------+-------+-------+")

    for r in range(9):
        print("| ", end="")

        for c in range(9):
            val = board[r][c]

            if val == 0:
                print(".", end=" ")
            else:
                print(val, end=" ")

            if (c + 1) % 3 == 0:
                print("| ", end="")

        print()

        if (r + 1) % 3 == 0:
            print("+-------+-------+-------+")


# RUN ONE PUZZLE
def run(filename):
    print("\n\n     ",filename)

    board = load_board(filename)
    solver = SudokuCSP(board)

    solution = solver.solve()

    solved_board = [[solution[(r, c)] for c in range(9)] for r in range(9)]

    print("\nSolved Sudoku:")
    print_board(solved_board)

    print("\nBacktrack Calls:", solver.backtrack_calls)
    print("Backtrack Failures:", solver.backtrack_failures)


# MAIN (RUN ALL FILES)
if __name__ == "__main__":
    files = ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]

    for f in files:
        run(f)



      easy.txt

Solved Sudoku:
+-------+-------+-------+
| 7 8 4 | 9 3 2 | 1 5 6 | 
| 6 1 9 | 4 8 5 | 3 2 7 | 
| 2 3 5 | 1 7 6 | 4 8 9 | 
+-------+-------+-------+
| 5 7 8 | 2 6 1 | 9 3 4 | 
| 3 4 1 | 8 9 7 | 5 6 2 | 
| 9 2 6 | 5 4 3 | 8 7 1 | 
+-------+-------+-------+
| 4 5 3 | 7 2 9 | 6 1 8 | 
| 8 6 2 | 3 1 4 | 7 9 5 | 
| 1 9 7 | 6 5 8 | 2 4 3 | 
+-------+-------+-------+

Backtrack Calls: 82
Backtrack Failures: 0


      medium.txt

Solved Sudoku:
+-------+-------+-------+
| 4 3 5 | 2 6 9 | 7 8 1 | 
| 6 8 2 | 5 7 1 | 4 9 3 | 
| 1 9 7 | 8 3 4 | 5 6 2 | 
+-------+-------+-------+
| 8 2 6 | 1 9 5 | 3 4 7 | 
| 3 7 4 | 6 8 2 | 9 1 5 | 
| 9 5 1 | 7 4 3 | 6 2 8 | 
+-------+-------+-------+
| 5 1 9 | 3 2 6 | 8 7 4 | 
| 2 4 8 | 9 5 7 | 1 3 6 | 
| 7 6 3 | 4 1 8 | 2 5 9 | 
+-------+-------+-------+

Backtrack Calls: 82
Backtrack Failures: 0


      hard.txt

Solved Sudoku:
+-------+-------+-------+
| 3 8 6 | 4 2 9 | 1 5 7 | 
| 1 7 4 | 8 6 5 | 2 3 9 | 
| 2 9 5 | 3 1 7 | 4 8 6 | 
+-------+---